## Численная оптимизация в логистических задачах: ДЗ

Совместим сразу два ДЗ в одно. Пусть есть классическая траспортная задача, которую надо решить двумя способами.

## 0. Генерация задания

In [1]:
import numpy as np

Зададим случайное число складов и магазинов:

In [2]:
N, M = np.random.randint(10, 100, 2)
N, M

(13, 83)

Заполним случайно массивы с количеством товара на складах и запросами магазинов:

**Кстати!** Вы же помните, что суммарное число запасов должно совпадать с суммарными запросами?

In [3]:
supply = np.random.randint(500, 1000)
storages = np.random.randint(1, 100, N)
demands = np.random.randint(1, 100, M)

full_demand = supply 
storages = storages / storages.sum() * supply 
demands = demands / demands.sum() * full_demand 

Сгенерируем случайно матрицу стоимости перевозок:

In [4]:
costs = np.random.randint(1, 20, size=(N, M))
costs

array([[ 9, 18, 15, ..., 10, 13, 12],
       [10,  9, 17, ...,  2,  6, 11],
       [ 6, 14, 13, ...,  5,  5,  8],
       ...,
       [ 3, 11,  3, ...,  9, 16, 16],
       [14,  3, 19, ...,  3, 19,  6],
       [ 4, 12, 12, ...,  1,  2, 13]])

Фух. Вроде все, можно приступать!

## 1. Постановка ЛП

Напишем честную постановку задачи Линейного программирования (ЛП) и попробуем решить задачу.

In [5]:
from pulp import *
from time import time

In [6]:
def make_problem(storages, demands, costs):
    N, M = len(storages), len(demands)
    
    prob = LpProblem("Prob", LpMinimize)
    
    #определяем переменные
    x = [[LpVariable(f"x_{i}_{j}", lowBound=0) for j in range(M)] for i in range(N)]
    
    #наша задача минимизировать стоимость перевозки
    prob += lpSum([x[i][j] * costs[i][j] for i in range(N) for j in range(M)]), "Total_Transport_Cost"
    
    #ограничение на количество товара со складов
    for i in range(N):
        prob += lpSum([x[i][j] for j in range(M)]) <= storages[i], f"Storage_Limit_{i}"
    
    #ограничение на количество товара получаемого магазинами со склада
    for j in range(M):
        prob += lpSum([x[i][j] for i in range(N)]) >= demands[j], f"Demand_Limit_{j}"
    
    return prob

In [7]:
problem = make_problem(storages, demands, costs)

st_time = time()
problem.solve()
print(f"Решение заняло {time() - st_time} сек")


Решение заняло 0.06449341773986816 сек


In [8]:
LpStatus[problem.status]

'Optimal'

Посмотрим на значение целевой функции:

In [9]:
problem.objective

9*x_0_0 + 18*x_0_1 + 2*x_0_10 + 8*x_0_11 + 9*x_0_12 + 17*x_0_13 + 13*x_0_14 + 7*x_0_15 + 10*x_0_16 + 11*x_0_17 + 10*x_0_18 + 11*x_0_19 + 15*x_0_2 + 4*x_0_20 + 15*x_0_21 + 12*x_0_22 + 1*x_0_23 + 10*x_0_24 + 1*x_0_25 + 12*x_0_26 + 9*x_0_27 + 11*x_0_28 + 18*x_0_29 + 14*x_0_3 + 13*x_0_30 + 4*x_0_31 + 1*x_0_32 + 5*x_0_33 + 12*x_0_34 + 4*x_0_35 + 15*x_0_36 + 15*x_0_37 + 10*x_0_38 + 17*x_0_39 + 13*x_0_4 + 14*x_0_40 + 7*x_0_41 + 1*x_0_42 + 19*x_0_43 + 14*x_0_44 + 13*x_0_45 + 1*x_0_46 + 17*x_0_47 + 8*x_0_48 + 14*x_0_49 + 16*x_0_5 + 6*x_0_50 + 13*x_0_51 + 4*x_0_52 + 14*x_0_53 + 19*x_0_54 + 16*x_0_55 + 10*x_0_56 + 1*x_0_57 + 11*x_0_58 + 17*x_0_59 + 13*x_0_6 + 10*x_0_60 + 8*x_0_61 + 7*x_0_62 + 9*x_0_63 + 7*x_0_64 + 19*x_0_65 + 17*x_0_66 + 17*x_0_67 + 3*x_0_68 + 12*x_0_69 + 2*x_0_7 + 13*x_0_70 + 13*x_0_71 + 1*x_0_72 + 16*x_0_73 + 16*x_0_74 + 6*x_0_75 + 12*x_0_76 + 7*x_0_77 + 7*x_0_78 + 6*x_0_79 + 9*x_0_8 + 10*x_0_80 + 13*x_0_81 + 12*x_0_82 + 12*x_0_9 + 3*x_10_0 + 11*x_10_1 + 15*x_10_10 + 16*x_10_11

## 2. Метод потенциалов

А теперь давайте для тех же данных реализуем метод потенциалов: будет ли он работать лучше / быстрее?

**Кстати**: для построения начальной конфигурации используйте метод северо-западного угла.

**Кстати**: для решения системы уравнений можно использовать фукнкцию np.linalg.solve(A, b), где A - квадратная матрица коэффициентов при переменных в уравнениях, а b - столбец правых частей уравнений. 

In [10]:
#метод северо-западного угла 
def northwest_corner(storages, demands):
    N, M = len(storages), len(demands)
    x = np.zeros((N, M))
    
    i, j = 0, 0
    while i < N and j < M:
        allocation = min(storages[i], demands[j])
        x[i, j] = allocation
        storages[i] -= allocation
        demands[j] -= allocation
        
        if storages[i] == 0:
            i += 1
        if demands[j] == 0:
            j += 1
            
    return x

In [11]:
#считаем потанциалы
def calculate_potentials(x, costs):
    N, M = x.shape
    u = np.full(N, np.nan)
    v = np.full(M, np.nan)
    
    # Устанавливаем u[0] = 0 для начала расчета
    u[0] = 0
    

    while np.any(np.isnan(u)) or np.any(np.isnan(v)):
        for i in range(N):
            for j in range(M):
                if x[i, j] > 0:  
                    if np.isnan(u[i]) and not np.isnan(v[j]):
                        u[i] = costs[i, j] - v[j]
                    elif not np.isnan(u[i]) and np.isnan(v[j]):
                        v[j] = costs[i, j] - u[i]
    
    return u, v

In [15]:
#оптимизация
def check_optimality(x, u, v, costs):
    """ Проверка и оптимизация решения """
    N, M = x.shape
    delta = np.zeros((N, M))  
    
    for i in range(N):
        for j in range(M):
            if x[i, j] == 0:  
                delta[i, j] = costs[i, j] - (u[i] + v[j])
    

    if np.all(delta >= 0):
        return x, True
    
    for _ in range(10):  
        min_delta_index = np.unravel_index(np.argmin(delta), delta.shape)
        i, j = min_delta_index
        if delta[i, j] >= 0:
            break
    
    return x, False

In [16]:
#реализация метода потенциалов
def solve_potentials(storages, demands, costs):
    x = northwest_corner(storages.copy(), demands.copy())
    
    iteration = 0
    max_iterations = 200
    optimal = False

    u, v = calculate_potentials(x, costs)
        
    while optimal == False and iteration < max_iterations:
        delta, optimal = check_optimality(x, u, v, costs)
        if np.all(delta < 0):  
            u, v = calculate_potentials(x, costs)
        iteration += 1    
    
    if iteration == max_iterations:
        print(f"Достигнуто максимальное количество итераций ({max_iterations})")
    
    return x

In [17]:
st_time = time()

res = solve_potentials(storages, demands, costs)

print(f"Решение заняло {time() - st_time} сек")

Достигнуто максимальное количество итераций (200)
Решение заняло 0.09329938888549805 сек


Какие выводы можно сделать?

Во-первых метод потанциалов подходит только для решения транспорных задач, в то время как ЛП подходит для любых задач, которые можно свести к линейным уравнениям.

Во-вторых, ЛП  легко автоматизировать с помощью готовых библиотек, чего нет у метода потанциалов

В-третьих, ЛП отработал быстрее